# 推理服务：Prefill、Decode、KV cache 与成本

解释推理服务的时间和内存分解，计算 KV 基线与每成功任务成本，用指标定位瓶颈。

Python 3.10+ / CPU / 无密钥。固定教学 fixture，不调用模型或服务。先读对应正文，预测成功与失败，再运行并修改一个边界。

In [ ]:
import math
def kv_bytes(layers,kv_heads,head_dim,tokens,bytes_each=2):
    if any(type(x) is not int or x<=0 for x in [layers,kv_heads,head_dim,tokens,bytes_each]):
        raise ValueError('positive integer dimensions required')
    return 2*layers*kv_heads*head_dim*tokens*bytes_each
size=kv_bytes(32,8,128,4096)
assert size==536870912  # 512 MiB，教学架构
costs=[1.0,2.0,3.0,4.0];success=[True,False,True,False]
per_success=sum(costs)/sum(success)
assert per_success==5.0  # 失败费用也计入
latencies=[1,2,3,4,20]
p95=sorted(latencies)[math.ceil(.95*len(latencies))-1]
assert p95==20  # 本例采用 nearest-rank，非所有工具的默认插值法
checks={'kv_baseline':True,'failed_cost_included':True,'tail_latency':True}
print(size//(1024**2),per_success,p95)

## 记录与继续

在下一格 my_explanation 写下预测、改动与实际观察；运行后下载 mechanism_report.json，导回本篇笔记。下一次先独立解释边界，再对照旧记录。

In [ ]:
import json
from pathlib import Path
my_explanation = ''
report={'experiment':'lesson-inference-serving', 'scope':'stdlib fixtures; no model/service', 'checks':checks, 'my_explanation':my_explanation}
Path('mechanism_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
print(json.dumps(report,ensure_ascii=False,indent=2))
